In [1]:
import pandas as pd #  pandas
import seaborn as sns # seaborn
import matplotlib.pyplot as plt # matplotlib.pyplot
from pprint import pprint # pprint para visualización metadatos
from ucimlrepo import fetch_ucirepo # se importa fetch_ucilrepo
import numpy as np # nump
import os
import seaborn as snst # seaborn
import statsmodels.api as sm #statsmodels
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Se carga el dataset DARWIN
darwin = fetch_ucirepo(id=732)
# X (features)
x = darwin.data.features

# Y (targets)
y = darwin.data.targets

# Se concatena features y target en un mismo dataframe 'df'
df = pd.concat([x,y], axis=1)

In [2]:
# Implementación base de datos alternativa
# ruta
ruta_darwin = os.path.expanduser("/Users/joseromerodegaetano/Desktop/DARWIN/DARWIN.csv") 

try:

    # Se carga el dataset
    df = pd.read_csv(ruta_darwin)

    x = df.drop(columns=['class'])
    y = df['class']

    # Se busca la col ID si existente y se filtra
    if 'ID' in x.columns:
        x = x.drop(columns=['ID'])
        print("Columna ID eliminada")

    print('Dataset DARWIN cargado correctamente')

except FileNotFoundError:
    print(f"No se encuentra el archivo .csv DARWIN en {ruta_darwin}")

Columna ID eliminada
Dataset DARWIN cargado correctamente


# PRUEBA ELIMINACIÓN DE VARIABLES

En este documento se repiten los análisis de ML eliminando las varaibles relacionadas, que presentan una alta correlación ( r > 0.90), para estudiar una posible mejora del modelo.

Las variables que se descartan por su correlación son:
  
- `total_time` (r = 0.97): es la suma de las variables `air_time` y `paper_time`.
- `mean_grmt` (r = 0.94): es la suma de las variables `grmt_in_air` y `gmrt_on_paper`.
- `mean_jerk_in_air` (r= 0.99): ya viene definido por `mean_acc_in_air`, el cual tiene ligeramente mejor correlación con la variable objetivo.
- `mean_speed_on_paper` (r=0.97): siempre muestra alta correlacón con `gmrt_on_paper`.

**Parece ser que la eliminación de estas variables no conlleva una mejora de los resultados, solo igual o peores métricas.**

In [3]:
# Se eliminan dichas columnas
# Nombres de las columnas
var_eliminar = ['mean_jerk_in_air',
                'mean_speed_on_paper']

# Lista completa con las columnas a eliminar
var_eliminar_final = [
    f"{var}{i}"
    for var in var_eliminar
    for i in range(1,26)
]

# Se eliminan las columnas del df original
df2 = df.drop(columns=var_eliminar_final, errors='ignore')

# Se verifica la correcta eliminación
print(f"Columnas eliminadas: {len(var_eliminar_final)}")
print(f"Nuevo tamaño del df: {df2.shape}")

Columnas eliminadas: 50
Nuevo tamaño del df: (174, 402)


## Preparación de los datos

In [4]:
# Se elimina la columna ID
# X = df (sin 'ID' ni 'class'), axis = 1 para cols
X = df2.drop(['ID', 'class'], axis = 1)

# Y contiene 'class'
Y = df2['class']

# Se codficia paciente (P) -> 1 y sano (H) -> 0
le = LabelEncoder()
Y = le.fit_transform(Y)

# Se dividen los datos en 80% para train y 20% para test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Se revisa la correcta división de los datos
print(f"Datos de entrenamiento {len(X_train)}")
print(f"Datos test {len(X_test)}")

Datos de entrenamiento 139
Datos test 35


## Estandarización de los datos

In [5]:
# StandardScaler()
scaler = StandardScaler()

# Se ajusta la estandarización con X_train
X_train_scaled = scaler.fit_transform(X_train)

# Se aplica la transformación al test
X_test_scaled = scaler.transform(X_test)

## 1. Regresión logística (LR)

### 1.1. Entrenamiento del modelo

Se entrena el modelo con los datos de entrenamiento (`X_train_scaled` y `Y_train`)

In [6]:
# Se crea el modelo
model_LR = LogisticRegression(max_iter=1000)

# Se entrena el modelo
model_LR.fit(X_train_scaled, Y_train)

print('El modelo se ha entrenado')

El modelo se ha entrenado


### 1.2. Evaluación del modelo

Se evalua el módelo con el informe de clasificación y la matriz de confusión

In [7]:
# Se realizan las predicciones
Y_pred = model_LR.predict(X_test_scaled)

# Informe de clasificación
print("--- INFORME DE CLASIFICACIÓN ---")
print(classification_report(Y_test, Y_pred))

# Matriz de confusión
print("--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(Y_test, Y_pred))

# Matriz
tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred).ravel()

# Métricas 
# Sensibilidad (capacidad para detectar los pacientes correctamente)
sensitivity = tp / (tp + fn)

# Especificdad (capacidad para dectectar los sanos correctamente)
specificity = tn / (tn + fp)

print(f"Sensibilidad: {sensitivity:.2f}")
print(f"Especificidad: {specificity:.2f}")

--- INFORME DE CLASIFICACIÓN ---
              precision    recall  f1-score   support

           0       0.69      0.65      0.67        17
           1       0.68      0.72      0.70        18

    accuracy                           0.69        35
   macro avg       0.69      0.68      0.68        35
weighted avg       0.69      0.69      0.69        35

--- MATRIZ DE CONFUSIÓN ---
[[11  6]
 [ 5 13]]
Sensibilidad: 0.72
Especificidad: 0.65


In [8]:
"""
C 
min value 0.001 
max value 5 
step 0.005
"""
c_values = np.arange(0.001, 5.005, 0.005)

# Diccionario de parámetros
param_grid = {'C': c_values}

# Búsqueda con 5 pliegues de validación cruzada
grid_search = GridSearchCV(
    LogisticRegression(max_iter=2000), 
    param_grid, 
    cv=5, 
    scoring='accuracy',
    verbose=1
)

# Se entrena el modelo
grid_search.fit(X_train_scaled, Y_train)

# Resultados
print(f"Mejor valor de C: {grid_search.best_params_['C']}")
print(f"Mejor Accuracy en CV: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
Y_pred_best = best_model.predict(X_test_scaled)

# Informe de clasificación
print("--- INFORME DE CLASIFICACIÓN ---")
print(classification_report(Y_test, Y_pred_best))

# Matriz de confusión
print("--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(Y_test, Y_pred_best))

# Matriz
tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred_best).ravel()

# Métricas 
# Sensibilidad (capacidad para detectar los pacientes correctamente)
sensitivity = tp / (tp + fn)

# Especificdad (capacidad para dectectar los sanos correctamente)
specificity = tn / (tn + fp)

print(f"Sensibilidad: {sensitivity:.2f}")
print(f"Especificidad: {specificity:.2f}")

Fitting 5 folds for each of 1001 candidates, totalling 5005 fits
Mejor valor de C: 0.011
Mejor Accuracy en CV: 0.9066
--- INFORME DE CLASIFICACIÓN ---
              precision    recall  f1-score   support

           0       0.65      0.65      0.65        17
           1       0.67      0.67      0.67        18

    accuracy                           0.66        35
   macro avg       0.66      0.66      0.66        35
weighted avg       0.66      0.66      0.66        35

--- MATRIZ DE CONFUSIÓN ---
[[11  6]
 [ 6 12]]
Sensibilidad: 0.67
Especificidad: 0.65
